# Week 57, Day 1: Feature maps and the kernel trick

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w057/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: An exclusive or, before and after a feature

The label is the exclusive or of the signs of two inputs. Fit penalised logistic regression on the raw inputs, then on the inputs plus their product x1 * x2, and compare test accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(571)

def xor_data(n):
    X = r.normal(size=(n, 2))
    return X, ((X[:, 0] > 0) != (X[:, 1] > 0)).astype(float)

def logreg(A, y, lam=1e-3):
    def f(w):
        p = np.clip(1 / (1 + np.exp(-A @ w)), 1e-12, 1 - 1e-12)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)) + lam * w @ w, A.T @ (p - y) / len(y) + 2 * lam * w
    return minimize(f, np.zeros(A.shape[1]), jac=True).x

X, y = xor_data(400); X_te, y_te = xor_data(2000)
raw = lambda X: np.c_[np.ones(len(X)), X]
lift = lambda X: ...
acc = lambda F, w, y: np.mean((F @ w > 0) == (y == 1))
acc_raw = acc(raw(X_te), logreg(raw(X), y), y_te)
acc_lift = acc(lift(X_te), logreg(lift(X), y), y_te)
print(f"raw inputs: test accuracy {acc_raw:.3f}; with the product feature: {acc_lift:.3f}")

In [ ]:
import numpy as np
def test_raw():
    "a linear classifier on the raw inputs is near chance"
    assert acc_raw < 0.65
def test_lifted():
    "adding the product feature makes it nearly perfect"
    assert acc_lift > 0.95

test_raw()
test_lifted()
print("All checks passed")

### Exercise: The trick, checked

Write phi2(x), the explicit degree-2 feature map for two-dimensional inputs: (1, √2 x1, √2 x2, x1², x2², √2 x1 x2). Check the kernel identity numerically. Then write n_features(d, q), the number of monomials of degree at most q in d variables, which is the binomial coefficient C(d + q, q).

In [ ]:
from math import comb

def phi2(x):
    x1, x2 = x
    return ...

def n_features(d, q):
    return ...

x, z = r.normal(size=2), r.normal(size=2)
print(f"explicit features: {phi2(x) @ phi2(z):.6f}; kernel: {(1 + x @ z) ** 2:.6f}")
for d, q in [(2, 2), (100, 2), (100, 3), (1000, 3)]:
    print(f"d = {d:4d}, degree {q}: {n_features(d, q):,} features against {d} multiplications for the kernel")

In [ ]:
import numpy as np
def test_phi():
    "phi2 lists the 6 scaled monomials for degree 2 in two dimensions"
    assert phi2(np.array([1.0, 2.0])).shape == (6,)
def test_equal():
    "phi2(x) · phi2(z) equals (1 + x · z)² for random vectors"
    x, z = r.normal(size=2), r.normal(size=2)
    assert abs(phi2(x) @ phi2(z) - (1 + x @ z) ** 2) < 1e-10
def test_count():
    "n_features counts monomials of degree up to q in d dimensions"
    assert n_features(2, 2) == 6 and n_features(100, 3) == 176851

test_phi()
test_equal()
test_count()
print("All checks passed")

### Exercise: Kernel or not?

Write gram(k, X). For 40 random points in three dimensions, compute the smallest eigenvalue of the Gram matrix for the Gaussian kernel, the degree-2 polynomial kernel, their sum, and the 'similarity' minus the Euclidean distance, which is not a kernel.

In [ ]:
def gram(k, X):
    return ...

Xp = r.normal(size=(40, 3))
kernels = {"gaussian": lambda a, b: np.exp(-0.5 * np.sum((a - b) ** 2)),
           "polynomial": lambda a, b: (1 + a @ b) ** 2,
           "sum": lambda a, b: np.exp(-0.5 * np.sum((a - b) ** 2)) + (1 + a @ b) ** 2,
           "negative distance": lambda a, b: -np.sqrt(np.sum((a - b) ** 2))}
min_eig = {name: np.linalg.eigvalsh(gram(k, Xp)).min() for name, k in kernels.items()}
for name, v in min_eig.items():
    print(f"{name:18s} smallest eigenvalue {v:.4f}")

In [ ]:
import numpy as np
def test_gram():
    "gram(k, X) returns the n x n matrix of kernel values"
    G = gram(lambda a, b: a @ b, np.eye(3)); assert np.allclose(G, np.eye(3))
def test_valid():
    "Gaussian, polynomial and their sum have no negative eigenvalues; the negative distance does"
    assert min_eig["gaussian"] > -1e-8 and min_eig["polynomial"] > -1e-8 and min_eig["sum"] > -1e-8 and min_eig["negative distance"] < -1e-3

test_gram()
test_valid()
print("All checks passed")